<a href="https://colab.research.google.com/github/mundundan-star/online-retail-customer-analytics/blob/main/Data_Engineering%20Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#!pip install boto3 openpyxl

In [ ]:
import pandas as pd
import numpy as np

import io
import boto3
from google.colab import userdata

aws_key = userdata.get("AWS_ACCESS_KEY_ID")
aws_secret = userdata.get("AWS_SECRET_ACCESS_KEY")

s3_client = boto3.client(
    "s3",
    aws_access_key_id = aws_key,
    aws_secret_access_key = aws_secret,
    region_name = "eu-north-1"
)

bucket_name = "sales-data-analytics-portfolio-2026"
file_key = "online+retail/Online Retail.xlsx"

print("Streaming Excel spreadsheet from Amazon S3...")

try:
    s3_object = s3_client.get_object(Bucket = bucket_name, Key = file_key)

    df = pd.read_excel(io.BytesIO(s3_object['Body'].read()))
    print("-> SuCcess! File successfully imported into Dataframe variable df")

    print("\nFirst 5 rows of the dataset:")
    print(df.head())
except Exception as e:
    print(f'\nPipeline broke! Error details {e}')

Streaming Excel spreadsheet from Amazon S3...


In [ ]:
from sqlalchemy import create_engine, text, inspect

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate']).dt.date

engine = create_engine(userdata.get("NEON_DATABASE_URL"))

drop_query = """
DROP TABLE IF EXISTS fact_online_retail
CASCADE;
"""
# Code drops existing table if exists and any dependent objects
with engine.begin() as conn:
    conn.execute(text(drop_query))

df.to_sql(name = 'fact_online_retail', con = engine, if_exists = 'replace', index = False)
print('Done! Data sent.')

Done! Data sent.


In [ ]:
query = """
    SELECT *
    FROM fact_online_retail;
"""

txn_df = pd.read_sql(query, con = engine)
print(txn_df.info())
txn_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  object 
 1   StockCode    541909 non-null  object 
 2   Description  540455 non-null  object 
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  object 
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 33.1+ MB
None


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01,3.39,17850.0,United Kingdom


In [ ]:
# 1.0 Formating InvoiceNo as interger and adding 'Cancelled' transaction flagging column

  # Assigning cancelled column
cancelled_transaction = txn_df['InvoiceNo'].astype("str").str.upper().str.contains("C")
txn_df['IsCancelled'] = np.where(cancelled_transaction, 1, 0)

# 2.0 Characterising stockcodes
numeric_list = list(range(10))
numeric_list = [str(x) for x in numeric_list]
unique_codes = txn_df['StockCode'].unique()
other_codes = []

for code in unique_codes:
    # Pull first 5 characters, as first 5 are expected to be numeric
    for x in code[0:5]:
        validation = ""
        # Looping through each of the first 5, when non-integer is detected, flag code as other code
        if x in numeric_list:
            validation = True
        else:
            validation = False
            # Checking if such a code is already in list to avoid duplicates and adding to list if doesn't exist
            if code not in other_codes:
                other_codes.append(code)

  # Assigning column to identity other codes
is_other_code = txn_df["StockCode"].isin(other_codes)
txn_df["IsOtherCode"] = np.where(is_other_code, 1, 0)

# 3.0 Assigning negative quantities as product returns and flagging negative prices
returned = txn_df["Quantity"] < 0
txn_df["IsReturned"] = np.where(returned, 1, 0)

neg_price = txn_df["UnitPrice"] < 0
txn_df["IsNegativePrice"] = np.where(neg_price, 1, 0)

# 4.0 Flagging null CustomerID values as "Guest"
guest = txn_df["CustomerID"].isna()
txn_df["IsGuest"] = np.where(guest, 1,0)

engine = create_engine(userdata.get("NEON_DATABASE_URL"))
txn_df.to_sql(name = "fact_online_retail", con = engine, if_exists = "replace", index = False)
print("Done! Data sent to Postgress Database and Table replaced")

In [ ]:

# Creating View of Clean Data

from sqlalchemy import text

query = """
CREATE OR REPLACE VIEW v_clean_sales_analytics AS
SELECT DISTINCT
    "CustomerID",
    "InvoiceNo",
    "InvoiceDate",
    "StockCode",
    "Description",
    "Quantity",
    "UnitPrice",
    ("Quantity" * "UnitPrice") AS "Revenue"
FROM fact_online_retail
WHERE "IsCancelled" = 0
    AND "IsReturned" = 0
    AND "IsGuest" = 0
    AND "IsNegativePrice" = 0;
"""
engine = create_engine(userdata.get("NEON_DATABASE_URL"))
with engine.connect() as conn:
    conn.execute(text(query))
    conn.commit()